# Walmart - Prédiction des ventes hebdomadaires

## Part 1 : EDA & Preprocessing

Objectif : explorer les données, nettoyer le dataset et préparer les features pour la modélisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

import warnings
warnings.filterwarnings('ignore')

## 1. Chargement et audit des données

In [ ]:
df = pd.read_csv("../data/raw/Walmart_Store_sales.csv")
print(f"Dataset : {df.shape[0]} lignes, {df.shape[1]} colonnes")
display(df.head())
df.info()

In [ ]:
df.describe()

In [ ]:
df['Weekly_Sales'].dropna().describe()

### 1.1 Nettoyage de la target (`Weekly_Sales`)

On ne fait jamais d'imputation sur la target → on drop les lignes manquantes.

In [ ]:
missing_target = df['Weekly_Sales'].isnull().sum()
print(f"Target manquante : {missing_target} lignes")

df = df.dropna(subset=['Weekly_Sales'])
print(f"Après suppression : {df.shape}")

### 1.2 Feature engineering : dates

Extraction de Year, Month, Day, DayOfWeek depuis la colonne `Date`, puis suppression de la colonne originale.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format="%d-%m-%Y")

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

df = df.drop("Date", axis=1)
display(df.head())

### 1.3 Suppression des outliers (règle 3 sigmas)

On exclut les valeurs hors $[\bar{X} - 3\sigma, \bar{X} + 3\sigma]$ pour Temperature, Fuel_Price, CPI et Unemployment.

In [ ]:
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

print(f"Avant outliers : {df.shape}")

for col in numeric_features:
    mean = df[col].mean()
    std = df[col].std()
    lower_bound = mean - 3 * std
    upper_bound = mean + 3 * std
    
    mask_outliers = (df[col] >= lower_bound) & (df[col] <= upper_bound)
    df = df.loc[mask_outliers | df[col].isnull(), :]
    
    print(f"  {col} : [{lower_bound:.2f}, {upper_bound:.2f}]")

print(f"Après outliers : {df.shape}")

### 1.4 Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Weekly_Sales'], bins=25, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution de Weekly_Sales")
axes[0].set_xlabel("Weekly Sales ($)")

sns.boxplot(x=df['Weekly_Sales'], ax=axes[1], color='steelblue')
axes[1].set_title("Boxplot Weekly_Sales")

plt.tight_layout()
plt.savefig("../assets/images/distribution_target.png", dpi=150, bbox_inches='tight')
plt.show()

## 2. Visualisation (EDA)

### 2.1 Matrice de corrélation

In [ ]:
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.savefig("../assets/images/correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Saisonnalité des ventes par mois

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Month', y='Weekly_Sales', data=df, palette="viridis")
plt.title("Ventes moyennes par mois")
plt.xlabel("Mois")
plt.ylabel("Ventes hebdomadaires moyennes")
plt.tight_layout()
plt.savefig("../assets/images/seasonality.png", dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Indicateurs économiques vs ventes

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

features_to_plot = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for i, feature in enumerate(features_to_plot):
    row, col = i // 2, i % 2
    sns.scatterplot(x=df[feature], y=df['Weekly_Sales'], ax=axs[row, col], color=colors[i], alpha=0.4)
    axs[row, col].set_title(f"Weekly_Sales vs {feature}")

plt.tight_layout()
plt.savefig("../assets/images/economic_indicators.png", dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing Scikit-Learn

### 3.1 Séparation X/Y et train/test split (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

Y = df['Weekly_Sales']
X = df.drop('Weekly_Sales', axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"X_train : {X_train.shape}, X_test : {X_test.shape}")

### 3.2 Pipeline de preprocessing (ColumnTransformer)

- **Numériques** (Temperature, Fuel_Price, CPI, Unemployment, Year, Month, Day, DayOfWeek) : imputation moyenne + StandardScaler
- **Catégorielles** (Store, Holiday_Flag) : imputation mode + OneHotEncoder (drop='first' pour éviter la colinéarité)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'DayOfWeek']
categorical_features = ['Store', 'Holiday_Flag']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), 
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

### 3.3 Application du preprocessing

Le `fit` se fait uniquement sur le train pour éviter le data leakage.

In [ ]:
# fit sur train uniquement
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print(f"X_train après preprocessing : {X_train.shape}")

## 4. Modèle Baseline : Régression Linéaire

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(X_train, Y_train)

### 4.1 Évaluation des performances

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

Y_train_pred = regressor.predict(X_train)
Y_test_pred = regressor.predict(X_test)

print("=== Régression Linéaire ===")
print(f"R2  Train : {r2_score(Y_train, Y_train_pred):.3f} | Test : {r2_score(Y_test, Y_test_pred):.3f}")
print(f"MAE Test  : {mean_absolute_error(Y_test, Y_test_pred):,.0f} $")
print(f"RMSE Test : {np.sqrt(mean_squared_error(Y_test, Y_test_pred)):,.0f} $")

### 4.2 Réel vs Prédit (test set)

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=Y_test, y=Y_test_pred, alpha=0.6)
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--')
plt.xlabel("Valeurs réelles")
plt.ylabel("Valeurs prédites")
plt.title("Réel vs Prédit - Régression Linéaire (test set)")
plt.tight_layout()
plt.savefig("../assets/images/real_vs_predicted.png", dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Interprétation des coefficients

In [ ]:
feature_names_out = preprocessor.get_feature_names_out()

coefs = pd.DataFrame(
    index=feature_names_out, 
    data=regressor.coef_, 
    columns=["Coefficient"]
)
coefs["Abs"] = coefs["Coefficient"].abs()
coefs_sorted = coefs.sort_values("Abs", ascending=False)

print("Top 10 features :")
display(coefs_sorted.head(10))

plt.figure(figsize=(10, 6))
sns.barplot(y=coefs_sorted.head(10).index, x="Coefficient", data=coefs_sorted.head(10), palette="magma")
plt.title("Top 10 features par coefficient")
plt.tight_layout()
plt.savefig("../assets/images/feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()

## 5. Régularisation

### 5.1 Ridge Regression

On ajoute une pénalité L2 ($\alpha \sum \beta_i^2$) pour réduire l'overfitting observé (écart train/test).

In [ ]:
from sklearn.linear_model import Ridge

# Test avec alpha=100 (volontairement élevé pour voir l'effet)
ridge_100 = Ridge(alpha=100)
ridge_100.fit(X_train, Y_train)

Y_test_pred_ridge100 = ridge_100.predict(X_test)

print("=== Ridge (alpha=100) ===")
print(f"R2  Train : {ridge_100.score(X_train, Y_train):.3f} | Test : {ridge_100.score(X_test, Y_test):.3f}")
print(f"MAE Test  : {mean_absolute_error(Y_test, Y_test_pred_ridge100):,.0f} $")
print(f"RMSE Test : {np.sqrt(mean_squared_error(Y_test, Y_test_pred_ridge100)):,.0f} $")
print("→ Alpha trop élevé : le modèle est trop pénalisé, d'où l'effondrement du R2.")

### 5.2 Optimisation de alpha par GridSearchCV (Ridge)

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

grid_ridge = GridSearchCV(Ridge(), param_grid=params, cv=5, scoring='r2')
grid_ridge.fit(X_train, Y_train)

best_ridge = grid_ridge.best_estimator_
Y_test_pred_best_ridge = best_ridge.predict(X_test)

print(f"=== Ridge optimisé (alpha={grid_ridge.best_params_['alpha']}) ===")
print(f"R2 CV moyen : {grid_ridge.best_score_:.3f}")
print(f"R2  Train : {best_ridge.score(X_train, Y_train):.3f} | Test : {best_ridge.score(X_test, Y_test):.3f}")
print(f"MAE Test  : {mean_absolute_error(Y_test, Y_test_pred_best_ridge):,.0f} $")
print(f"RMSE Test : {np.sqrt(mean_squared_error(Y_test, Y_test_pred_best_ridge)):,.0f} $")

### 5.3 Lasso Regression (pénalité L1)

Contrairement à Ridge (L2), Lasso peut mettre des coefficients exactement à 0 → sélection de features automatique.

In [ ]:
from sklearn.linear_model import Lasso

params_lasso = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 500, 1000]}

grid_lasso = GridSearchCV(Lasso(max_iter=10000), param_grid=params_lasso, cv=5, scoring='r2')
grid_lasso.fit(X_train, Y_train)

best_lasso = grid_lasso.best_estimator_
Y_test_pred_lasso = best_lasso.predict(X_test)

print(f"=== Lasso optimisé (alpha={grid_lasso.best_params_['alpha']}) ===")
print(f"R2 CV moyen : {grid_lasso.best_score_:.3f}")
print(f"R2  Train : {best_lasso.score(X_train, Y_train):.3f} | Test : {best_lasso.score(X_test, Y_test):.3f}")
print(f"MAE Test  : {mean_absolute_error(Y_test, Y_test_pred_lasso):,.0f} $")
print(f"RMSE Test : {np.sqrt(mean_squared_error(Y_test, Y_test_pred_lasso)):,.0f} $")

# Coefficients à 0 (features éliminées par Lasso)
n_zeros = (best_lasso.coef_ == 0).sum()
print(f"\nFeatures éliminées par Lasso : {n_zeros}/{len(best_lasso.coef_)}")

## 6. Comparaison des modèles

In [ ]:
results = pd.DataFrame({
    'Modèle': ['Régression Linéaire', f'Ridge (alpha={grid_ridge.best_params_["alpha"]})', f'Lasso (alpha={grid_lasso.best_params_["alpha"]})'],
    'R2 Train': [
        r2_score(Y_train, regressor.predict(X_train)),
        best_ridge.score(X_train, Y_train),
        best_lasso.score(X_train, Y_train)
    ],
    'R2 Test': [
        r2_score(Y_test, Y_test_pred),
        best_ridge.score(X_test, Y_test),
        best_lasso.score(X_test, Y_test)
    ],
    'MAE Test': [
        mean_absolute_error(Y_test, Y_test_pred),
        mean_absolute_error(Y_test, Y_test_pred_best_ridge),
        mean_absolute_error(Y_test, Y_test_pred_lasso)
    ],
    'RMSE Test': [
        np.sqrt(mean_squared_error(Y_test, Y_test_pred)),
        np.sqrt(mean_squared_error(Y_test, Y_test_pred_best_ridge)),
        np.sqrt(mean_squared_error(Y_test, Y_test_pred_lasso))
    ]
})

results['R2 Train'] = results['R2 Train'].map('{:.3f}'.format)
results['R2 Test'] = results['R2 Test'].map('{:.3f}'.format)
results['MAE Test'] = results['MAE Test'].map('${:,.0f}'.format)
results['RMSE Test'] = results['RMSE Test'].map('${:,.0f}'.format)

display(results)

## 7. Conclusion

**Meilleur modèle** : Ridge régularisé — R2 test ~0.895, légèrement meilleur que la régression linéaire simple (0.891).

**Observations** :
- L'écart train/test modéré (0.977 vs 0.891) indique un léger overfitting, corrigé partiellement par Ridge
- Le best alpha très faible (0.1) montre que le modèle n'avait pas besoin de forte régularisation
- Lasso confirme cette tendance : peu de features sont réellement éliminées
- Le Store est de loin la feature la plus prédictive (effet taille/localisation du magasin)

**Limites** :
- Dataset très petit (131 lignes après nettoyage) → résultats sensibles au split
- OneHotEncoding de Store crée beaucoup de colonnes pour peu d'observations
- Les indicateurs économiques (CPI, Unemployment) montrent peu de corrélation linéaire avec les ventes